In [ ]:
import pandas as pd
import os
import jaydebeapi
import jpype
from datetime import datetime
from dateutil.relativedelta import relativedelta
from dotenv import load_dotenv, find_dotenv
import warnings

warnings.filterwarnings("ignore")
load_dotenv(find_dotenv())

# ================================
# AUTO CONFIG
# ================================
today = datetime.today()

campaign_year = today.year
campaign_month = today.month

# Previous month
data_date = today - relativedelta(months=1)

data_month = data_date.month
data_year = data_date.year

# ================================
# PERIODS
# ================================
start_of_month = datetime(data_year, data_month, 1)

end_of_month = (
    start_of_month
    + relativedelta(months=1)
    - relativedelta(days=1)
)

start_ts = start_of_month.strftime("%Y-%m-%d 00:00:00")
end_ts = end_of_month.strftime("%Y-%m-%d 23:59:59")

current_start = datetime(today.year, today.month, 1)

current_end = (
    current_start
    + relativedelta(months=1)
    - relativedelta(days=1)
)

current_next = (
    current_start
    + relativedelta(months=1)
)

vpodsa_date_current = current_end.strftime("%Y%m%d")
vpadsa_next_current = current_next.strftime("%Y%m%d")

print("\n================ AUTO Wallonie CONFIG ================")
print("Campaign month:", campaign_month)
print("Data month:", data_month)
print("DB snapshot month:", today.month)

# ================================
# HELPERS
# ================================
def anatella_cast(series):

    return (
        pd.to_numeric(series, errors="coerce")
        .astype("Int64")
        .astype(str)
        .str.replace("<NA>", "", regex=False)
        .str.replace(".0", "", regex=False)
        .str.strip()
    )

def normalize_exid(x):

    if pd.isna(x):
        return None

    try:
        return str(x).strip()
    except:
        return None

def clean_exid(series):

    return (
        series
        .apply(
            lambda x: ''.join(x)
            if isinstance(x, tuple)
            else str(x)
        )
        .str.strip()
    )

# ================================
# CONNECTION
# ================================
def connect():

    jar_path = r"C:\db2\db2jcc4.jar"

    if not jpype.isJVMStarted():

        jpype.startJVM(
            jpype.getDefaultJVMPath(),
            "-Xmx4g",
            "-Djava.class.path=" + jar_path
        )

    return jaydebeapi.connect(
        "com.ibm.db2.jcc.DB2Driver",
        "jdbc:db2://s998lp1dbbi01.jablux.cpc998.be:50004/ods500",
        ["m509psao", os.getenv("DB_PASSWORD")]
    )

conn = connect()

# ================================
# DB QUERY
# ================================
query = f"""
SELECT
    p.EXIDSA,
    p.PROVSA,
    v.VPV1SV,
    p.NAIDSA,

    CASE
        WHEN s.EXIDSA IS NOT NULL THEN 1
        ELSE 0
    END AS OPT_OUT_FLAG

FROM ODS509.ODS_PSTATA p

LEFT JOIN ODS509.ODS_PSTATV v
    ON p.EXIDSA = v.EXIDSV

LEFT JOIN (
    SELECT DISTINCT
        REPLACE(TARGETID, ',', '') AS EXIDSA
    FROM ODS509.V_ISE_ODS_WWWSIGNAL
    WHERE SIGNALID = 'CampagneCC'
    AND INACTIVE = 0
) s
    ON p.EXIDSA = s.EXIDSA

WHERE (

    p.VPODSA >= {vpodsa_date_current}

    AND p.VPACSA <> 'O'

    AND (
        (
            p.VPACSA = 'T'
            AND p.VPADSA <= {vpadsa_next_current}
        )

        OR

        (
            p.VPACSA IN ('A','B','C','L')
            AND p.VPADSA < {vpadsa_next_current}
        )
    )

    AND p.IV00SA = 'B'
)
"""

print("\nLoading DB query...")

df = pd.read_sql(query, conn)

df.columns = [str(c).upper() for c in df.columns]

print(
    "\nDB RAW:",
    len(df),
    "| Unique:",
    df["EXIDSA"].nunique()
)

# ================================
# CLEAN DB
# ================================
df["EXID"] = (
    df["EXIDSA"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.strip()
)

df["VPV1SV"] = (
    df["VPV1SV"]
    .astype(str)
    .str.strip()
)

# FILTER
df = df[
    (df["VPV1SV"] >= "000")
    &
    (
        (df["VPV1SV"] < "750")
        |
        (df["VPV1SV"] > "755")
    )
]

# GROUP
df = df.groupby(
    "EXID",
    as_index=False
).agg({
    "PROVSA": "first",
    "VPV1SV": "first",
    "OPT_OUT_FLAG": "max",
    "NAIDSA": "first"
})

print("DB FINAL:", len(df))
print("UNIQUE EXID:", df["EXID"].nunique())

# ================================
# POP
# ================================
pop = pd.read_excel(
    r"G:\Studies\Cellule Etudes\Reporting\Marketing\List reminder colorectal cancer screening\POP_ELIGIBLE_PARTENAMUT_2026_NEW.xlsx"
)

pop.columns = [str(c).upper() for c in pop.columns]

pop["EXID"] = anatella_cast(
    pop["EXID"]
)

pop = pop.drop_duplicates(
    subset=["EXID"]
)

print("POP:", len(pop))

# ================================
# BASE
# ================================
base = pop.merge(
    df,
    on="EXID",
    how="inner"
)

print("BASE:", len(base))

# ================================
# OPT OUT COUNT
# ================================
opted_out_count = (
    pd.to_numeric(
        base["OPT_OUT_FLAG"],
        errors="coerce"
    ) == 1
).sum()

print("OPTED OUT MEMBERS:", opted_out_count)

# ================================
# FILTER BRUSSELS
# ================================
base = base[
    (
        pd.to_numeric(
            base["PROVSA"],
            errors="coerce"
        ) == 10
    )
    &
    (
        pd.to_numeric(
            base["OPT_OUT_FLAG"],
            errors="coerce"
        ) == 0
    )
]

print(
    "BASE FILTERED "
    "(after opt out + province filter):",
    len(base)
)

# ================================
# MONTH OF BIRTH
# ================================
base["MOIS_NAISS"] = (
    base["NAIDSA"]
    .astype(str)
    .str[4:6]
    .astype(int)
)

# ================================
# EBOX READ
# ================================
ebox_read = pd.read_sql("""
SELECT DISTINCT BENEFICIARYEXID
FROM ODS509.V_INF_ODS_DOCUMENT
WHERE ORGANIZATION = '509'
AND "READ" = 1
AND OUTPUTCHANNELS = 'infobox'
AND READDATE >= ADD_MONTHS(CURRENT_DATE, -18)
AND READDATE < DATE '9999-01-01'
""", conn)

ebox_read["user_exid"] = clean_exid(
    ebox_read["BENEFICIARYEXID"]
)

ebox_read["user_exid"] = anatella_cast(
    ebox_read["user_exid"]
)

ebox_read["user_exid"] = (
    ebox_read["user_exid"]
    .apply(normalize_exid)
    .str.upper()
)

print("READ count:", len(ebox_read))

# ================================
# NOT READ
# ================================
not_read = pd.read_sql(f"""
SELECT DISTINCT BENEFICIARYEXID
FROM ODS509.V_INF_ODS_DOCUMENT
WHERE ORGANIZATION = '509'
AND "READ" = 0
AND RECEIVEDATE BETWEEN TIMESTAMP '{start_ts}'
AND TIMESTAMP '{end_ts}'
AND NAME LIKE 'NMKTCBI%'
AND NOTIFICATIONREQUESTSTATUS = 'DELIVERED'
""", conn)

not_read["EXID"] = clean_exid(
    not_read["BENEFICIARYEXID"]
)

not_read["EXID"] = anatella_cast(
    not_read["EXID"]
)

not_read["EXID"] = (
    not_read["EXID"]
    .apply(normalize_exid)
    .str.upper()
)

not_read = (
    not_read[["EXID"]]
    .drop_duplicates()
)

print("NOT READ count:", len(not_read))

# ================================
# NORMALIZE
# ================================
base["EXID"] = (
    base["EXID"]
    .apply(normalize_exid)
    .str.upper()
)

# ================================
# VISITED / NOT VISITED
# ================================
ebox_active_exids = set(
    ebox_read["user_exid"]
)

visited = (
    base[
        base["EXID"].isin(ebox_active_exids)
    ]
    .drop_duplicates("EXID")
)

not_visited = (
    base[
        ~base["EXID"].isin(ebox_active_exids)
    ]
    .drop_duplicates("EXID")
)

# ================================
# MONTH FILTER
# ================================
visited_month = (
    visited[
        visited["MOIS_NAISS"] == data_month
    ][["EXID"]]
)

not_visited_month = (
    not_visited[
        not_visited["MOIS_NAISS"] == data_month
    ][["EXID"]]
)

print("NOT VISITED:", len(not_visited_month))
print("VISITED:", len(visited_month))

# ================================
# FINAL
# ================================
paper_send_total = pd.concat([
    not_visited_month,
    not_read
]).drop_duplicates(subset=["EXID"])

# ================================
# VALIDATION
# ================================
print("\n================ VALIDATION ================")

print("BASE:", len(base))
print("OPTED OUT:", opted_out_count)
print("eBox send:", len(visited_month))
print("Paper send:", len(paper_send_total))
print("  -- NO EBOX:", len(not_visited_month))
print("  -- NOT READ:", len(not_read))

# ================================
# EXPORT
# ================================
export_path = r"G:\Studies\Cellule Etudes\Reporting\Marketing\List reminder colorectal cancer screening"

visited_month.to_excel(
    os.path.join(
        export_path,
        f"Brussels_eBox_Send_{campaign_month:02d}_{campaign_year}.xlsx"
    ),
    index=False
)

paper_send_total.to_excel(
    os.path.join(
        export_path,
        f"Brussels_Paper_Send_{campaign_month:02d}_{campaign_year}.xlsx"
    ),
    index=False
)

print("\nExport Brussels completed successfully.")


================ AUTO CONFIG ================
Campaign month: 5
Data month: 4
DB snapshot month: 5

Loading DB query...

DB RAW: 1392505 | Unique: 1392505
DB FINAL: 1366306
UNIQUE EXID: 1366306
POP: 40338
BASE: 39948
OPTED OUT MEMBERS: 50
BASE FILTERED (after opt out + province filter): 39628
READ count: 899001
NOT READ count: 1607
NOT VISITED: 1396
VISITED: 1925

================ VALIDATION ================
BASE: 39628
OPTED OUT: 50
eBox send: 1925
Paper send: 3003
  -- NO EBOX: 1396
  -- NOT READ: 1607

Export Brussels completed successfully.
